In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12,5)

In [ ]:
# Load Data
df = pd.read_csv("../data/raw/online_retail_II.csv", dtype={'Invoice': str, 'StockCode': str, 'Customer ID': str}, parse_dates=['InvoiceDate'])

# Rename 'Customer ID' to 'CustomerID' --> easier to work with
df = df.rename(columns={'Customer ID':'CustomerID'})
    
print(f'Shape {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

Shape 1,067,371 rows x 8 columns


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [12]:
# Time range & basic counts
print('Date range')
print(f'    From:   {df['InvoiceDate'].min()}')
print(f'    To:     {df['InvoiceDate'].max()}')
print(f'    Span:   {(df['InvoiceDate'].max() - df['InvoiceDate'].min()).days} days')

print('\nBasic Counts')
print(f'Invoices:   {df['Invoice'].nunique():,}')
print(f'Customers:  {df['CustomerID'].nunique():,}')
print(f'Products:   {df['StockCode'].nunique():,}')
print(f'Countries:  {df['Country'].nunique()}')

Date range
    From:   2009-12-01 07:45:00
    To:     2011-12-09 12:50:00
    Span:   738 days

Basic Counts
Invoices:   53,628
Customers:  5,942
Products:   5,305
Countries:  43


In [14]:
# Check missing value
missing = df.isnull().sum()
missing_pct = ((missing / len(df) * 100)).round(2)
pd.DataFrame({'missing': missing, 'percentage': missing_pct}).sort_values('missing', ascending=False)

,missing,percentage
CustomerID,243007,22.77
Description,4382,0.41
StockCode,0,0.00
Invoice,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00


In [21]:
# Analyze cancellation 
df['is_cancelation'] = df['Invoice'].str.startswith('C')

cancel_summary = df.groupby('is_cancelation').agg(
    rows=('Invoice', 'size'),
    unique_invoices=('Invoice', 'nunique'),
    total_quantity=('Quantity', 'sum')
).rename(index={False: 'Sale', True: 'Cancellation'})

print(cancel_summary)
print(f'\nCancellation rows share: {df['is_cancelation'].mean() * 100:.2f}%')

                   rows  unique_invoices  total_quantity
is_cancelation                                          
Sale            1047877            45336        11099484
Cancellation      19494             8292         -490992

Cancellation rows share: 1.83%


In [22]:
# Quantity & Price distribution - finding the weird stuff
print('Quantity:')
print(df['Quantity'].describe())
print()
print('Price:')
print(df['Price'].describe())

Quantity:
count   1,067,371.00
mean            9.94
std           172.71
min       -80,995.00
25%             1.00
50%             3.00
75%            10.00
max        80,995.00
Name: Quantity, dtype: float64

Price:
count   1,067,371.00
mean            4.65
std           123.55
min       -53,594.36
25%             1.25
50%             2.10
75%             4.15
max        38,970.00
Name: Price, dtype: float64


In [25]:
# How many rows have suspicous value?
print("Suspicous value count:")
print(f'Quantity == 0 : {(df['Quantity'] == 0).sum():,}')
print(f'Quantity < 0 : {(df['Quantity'] < 0).sum():,} (likely cancellation)')
print(f'Quantity > 1000 : {(df['Quantity'] > 1000).sum():,} (bulk or error)') 
print(f'Price == 0 : {(df['Price'] == 0).sum():,}')
print(f'Price > 100 : {(df['Price'] > 100).sum():,}')
print(f'Price < 0 : {(df['Price'] < 0).sum():,}')

Suspicous value count:
Quantity == 0 : 0
Quantity < 0 : 22,950 (likely cancellation)
Quantity > 1000 : 352 (bulk or error)
Price == 0 : 6,202
Price > 100 : 1,946
Price < 0 : 5


In [26]:
# Look at the most extreme quantity outliers
df.nlargest(10, 'Quantity')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'CustomerID', 'Country']]

,Invoice,StockCode,Description,Quantity,Price,CustomerID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446.0,United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346.0,United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,13902.0,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,13902.0,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,13902.0,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,13902.0,Denmark
1027583,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00,13256.0,United Kingdom
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,13902.0,Denmark
192197,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,0.00,NaN,United Kingdom
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,17940.0,United Kingdom


In [28]:
# Look at the zero-price rows - why are they?
df[df['Price'] == 0]['Description'].value_counts().head(15)

Description
check                           162
?                                92
damages                          84
damaged                          81
found                            28
missing                          27
sold as set on dotcom            20
Damaged                          17
adjustment                       16
OWL DOORSTOP                     15
POLYESTER FILLER PAD 45x45cm     12
dotcom                           12
amazon                           11
POLYESTER FILLER PAD 40x40cm     10
IVORY KITCHEN SCALES             10
Name: count, dtype: int64